# Ruido Ambiental — Ciclo estadístico (L_Aeq,T, Res. 627/2006)

> **Bloque:** B — Transversal temática | alimenta: gestión de riesgo, ordenamiento territorial
> **Variable focal:** Nivel de presión sonora L_Aeq,T (dB(A))
> **Fuente:** sin conector automatizado hoy (ver sección "Datos" — este notebook usa datos sintéticos)
> **Objetivo:** EDA → Descriptiva → Inferencial → cumplimiento normativo (Tabla 2, Art. 17)

---

## Contexto institucional

Colombia no tiene una red nacional de monitoreo continuo de ruido ambiental comparable a la RMCAB/SIATA de calidad del aire. El control de ruido es responsabilidad de las **autoridades ambientales regionales (CARs) y las Secretarías Distritales/Municipales de Ambiente**, típicamente mediante mediciones puntuales (sonómetro) ante quejas o licenciamiento, no series continuas. La **Ley 2450 de 2025** ("Ley contra el Ruido") es el desarrollo normativo más reciente: ordena a MinAmbiente y MinSalud expedir una reglamentación técnica actualizada dentro de 18 meses desde marzo de 2025 (venciendo hacia septiembre de 2026) — ver `config.NORMA_FUENTES["NORMA_RUIDO"]` para el estado de verificación más reciente de si esa reglamentación ya se expidió.

| Actor | Rol |
|---|---|
| MinAmbiente | Regulación técnica (Res. 627/2006, reglamentación derivada de Ley 2450/2025) |
| MinSalud | Co-responsable de la reglamentación derivada de Ley 2450/2025 |
| CARs / Secretarías de Ambiente | Mediciones puntuales, sanciones, mapas de ruido locales |
| MinTransporte / MinDefensa | Fuentes móviles y ruido de infraestructura (Ley 2450/2025) |

---

## Normativa colombiana aplicable — Tabla 2 (Art. 17, Res. 627/2006)

Estándares máximos permisibles de **nivel de ruido ambiental** en dB(A), horario diurno (7:01-21:00) y nocturno (21:01-7:00). Viven en `config.NORMA_RUIDO` — **nunca hardcodear estos valores en el notebook**.

| Sector | Diurno (dB(A)) | Nocturno (dB(A)) |
|---|---|---|
| A — Hospitales, bibliotecas, guarderías | 55 | 45 |
| B — Residencial, hotelería, universidades | 65 | 50 |
| C — Industrial | 75 | 70 |
| C — Comercial | 70 | 55 |
| C — Oficinas | 65 | 50 |
| C — Espectáculos/vías (incl. troncales) | 80 | 70 |
| D — Rural/suburbana, parques naturales | 55 | 45 |

> **Unidad real:** L_Aeq,T — nivel continuo equivalente ponderado A, **no** un valor instantáneo. T=14h para el intervalo diurno, T=10h para el nocturno (Art. 15).
>
> **Excepción real (Art. 17, Parágrafo Segundo):** si el nivel se supera por fuentes **naturales sin intervención humana** (cascadas, fauna, etc. — relevante sobre todo en sector D), el estándar no aplica de la forma usual. `ruido_exceedance_report()` NO detecta esto automáticamente — es una aserción explícita del analista (`natural_noise=True`), no un análisis del dato.
>
> **No se usa la Tabla 1** (Art. 9, emisión de una fuente puntual aislada) — este notebook trabaja con mediciones continuas de ruido ambiental en una zona.


## 0. Setup

Se importan los módulos disponibles hoy para esta línea temática. A diferencia de calidad del aire, ruido ambiental **no tiene todavía**: conector de datos automatizado, módulo de EDA propio, ni modelos predictivos con evidencia empírica en este repo — se listan igual los módulos genéricos del ciclo estadístico que sí aplican a cualquier serie de tiempo ambiental.

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from estadistica_ambiental.io.validators import validate, PHYSICAL_RANGES
from estadistica_ambiental.eda.quality import assess_quality
from estadistica_ambiental.descriptive.univariate import summarize
from estadistica_ambiental.inference.stationarity import stationarity_report
from estadistica_ambiental.inference.trend import mann_kendall
from estadistica_ambiental.inference.intervals import ruido_exceedance_report
from estadistica_ambiental.config import NORMA_RUIDO, NORMA_FUENTES

print("Setup OK")
print("Sectores disponibles:", sorted({k.rsplit('_', 1)[0] for k in NORMA_RUIDO}))
print("Estado de verificación normativa:", NORMA_FUENTES["NORMA_RUIDO"]["fecha_verificacion"])

## 1. Datos

**¿Qué queremos saber aquí?**
- ☐ ¿De qué sector (Tabla 2) y período provienen los datos?
- ☐ ¿La resolución temporal alcanza para calcular L_Aeq,T real (idealmente sub-horaria)?
- ☐ ¿Hay cobertura suficiente de ambas ventanas horarias (diurna y nocturna) por día?

**Estado real de las fuentes (a diferencia de calidad del aire, sin conector hoy):**

| Fuente | Estado | Notas |
|---|---|---|
| CARs / Secretarías de Ambiente | Manual | Mediciones puntuales por queja/licenciamiento, no series continuas publicadas |
| SDA Bogotá | Manual | Mapas de ruido puntuales (no serie temporal descargable vía API) |
| `io/connectors.py` | **No implementado** | No existe `load_ruido_*` -- ver issue de seguimiento si se necesita |

> Sin una fuente pública descargable identificada, este notebook usa una serie **sintética** representativa (ver celda de código) para demostrar el flujo completo. Sustituir por datos reales apenas se identifique una fuente (ver sección 9).

In [ ]:
# Serie sintética de nivel de ruido (dB(A)) sub-horaria, sector B (residencial)
# Patron: base diurna mas alta que nocturna (trafico/actividad humana), con
# ruido gaussiano y algunos picos (eventos puntuales, ej. obras/eventos).
np.random.seed(42)
SECTOR = "sector_b"

fechas = pd.date_range("2024-01-01", "2024-01-15", freq="30min", inclusive="left")
hora = fechas.hour + fechas.minute / 60.0
es_diurno = (hora >= 7 + 1/60) & (hora <= 21.0)

base = np.where(es_diurno, 58.0, 47.0)
ruido_medicion = base + np.random.normal(0, 3, len(fechas))
# Picos ocasionales (eventos puntuales, ~2% de las mediciones)
picos_idx = np.random.choice(len(fechas), int(len(fechas) * 0.02), replace=False)
ruido_medicion[picos_idx] += np.random.uniform(10, 20, len(picos_idx))
ruido_medicion = np.clip(ruido_medicion, 20, 120)

df = pd.DataFrame({"fecha": fechas, "ruido": ruido_medicion.round(1)})
print(f"Dataset sintético: {df.shape[0]:,} mediciones cada 30 min | {df['fecha'].min()} → {df['fecha'].max()}")
df.head()

## 2. Validación y EDA

**¿Qué queremos saber aquí?**
- ☐ ¿Hay valores físicamente imposibles (< 0 o > 140 dB(A), umbral de dolor)?
- ☐ ¿Hay gaps temporales que dejen una ventana horaria (diurna/nocturna) sin datos suficientes ese día?
- ☐ ¿Qué distribución tiene la serie? (Se espera aproximadamente normal en escala dB, con cola derecha por eventos puntuales)

`validate()` usa el rango físico `PHYSICAL_RANGES["ruido"] = (0.0, 140.0)` -- 0 dB(A) es el umbral de audición humana, 140 dB(A) el umbral de dolor.

In [ ]:
val = validate(df, date_col="fecha")
print(val.summary())

In [ ]:
quality = assess_quality(df, date_col="fecha")
print(quality.summary())

## 3. Visualización

**¿Qué queremos saber aquí?**
- ☐ ¿Se distingue visualmente el patrón diurno/nocturno?
- ☐ ¿Con qué frecuencia se acercan o superan las líneas de la Tabla 2 para el sector elegido?

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(df["fecha"], df["ruido"], lw=0.6, color="#1a5276")
ax.axhline(NORMA_RUIDO[f"{SECTOR}_dia"], color="orange", ls="--", lw=1,
           label=f"Umbral diurno {SECTOR} ({NORMA_RUIDO[f'{SECTOR}_dia']} dB(A))")
ax.axhline(NORMA_RUIDO[f"{SECTOR}_noche"], color="red", ls="--", lw=1,
           label=f"Umbral nocturno {SECTOR} ({NORMA_RUIDO[f'{SECTOR}_noche']} dB(A))")
ax.set_ylabel("dB(A)")
ax.set_title(f"Nivel de ruido — mediciones sub-horarias (sector {SECTOR})")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 4. Estadística descriptiva

**¿Qué queremos saber aquí?**
- ☐ ¿Cuál es la media/mediana de las mediciones crudas? (Ojo: esto NO es lo mismo que L_Aeq,T -- ver sección 5)
- ☐ ¿Qué tan dispersa es la serie?

> **Advertencia de dominio:** la media aritmética de mediciones en dB **no** es el L_Aeq,T normativo -- el nivel continuo equivalente es un promedio energético (Art. 4), sistemáticamente más alto que la media aritmética cuando hay eventos puntuales. `ruido_exceedance_report()` calcula el promedio correcto; esta sección es solo descriptiva exploratoria.

In [ ]:
summarize(df[["ruido"]])

## 5. Cumplimiento normativo — `ruido_exceedance_report()`

Calcula L_Aeq (Art. 4, promedio **energético**, no aritmético) por día y ventana horaria (diurna/nocturna, Art. 2) y lo compara contra el estándar del sector (Tabla 2, Art. 17).

In [ ]:
df_idx = df.set_index("fecha")["ruido"]
reporte = ruido_exceedance_report(df_idx, sector=SECTOR)
print(f"{len(reporte)} filas (día × horario) | {reporte['excede'].sum()} en excedencia")
reporte.head(10)

In [ ]:
# Resumen: % de ventanas (día x horario) en excedencia, por horario
resumen = reporte.groupby("horario")["excede"].agg(["sum", "count"])
resumen["pct"] = (resumen["sum"] / resumen["count"] * 100).round(1)
print(resumen)

### Excepción por fuente natural (Art. 17, Parágrafo Segundo)

Si el analista determina (fuera de este notebook, con criterio técnico/de campo) que una excedencia se debe a una fuente natural sin intervención humana, pasar `natural_noise=True` -- las filas quedan marcadas como **no evaluadas**, no como "cumple" ni como "excede" con una excepción tácita:

In [ ]:
reporte_natural = ruido_exceedance_report(df_idx, sector="sector_d", natural_noise=True)
reporte_natural[["dia_ruido", "horario", "excede", "nota"]].head(3)

## 6. Inferencial

**¿Qué queremos saber aquí?**
- ☐ ¿Es la serie estacionaria? (mismo criterio ADF+KPSS que el resto del repo, ADR-004 -- aplica igual a ruido que a cualquier otra serie ambiental)
- ☐ ¿Hay tendencia significativa en el nivel de ruido a lo largo del período?

Estos dos análisis usan los mismos módulos genéricos (`inference/stationarity.py`, `inference/trend.py`) que el resto de las líneas temáticas -- no hay nada específico de ruido acá, por eso no se documentan hallazgos empíricos propios todavía (a diferencia de calidad del aire, que sí tiene evidencia real acumulada de sesiones anteriores).

In [ ]:
stationarity_report(df_idx)

In [ ]:
mk = mann_kendall(df_idx)
print(f"Tendencia: {mk['trend']} | p={mk['pval']:.4f} | slope={mk['slope']:.4f} dB(A)/medición")

## 7. Guardrails y supuestos metodológicos
<!-- GUARDRAILS: ruido_ambiental -->

### Supuestos comunes (todas las líneas)

- **Normas oficiales:** usar `config.NORMA_RUIDO` -- nunca umbrales hardcodeados en el notebook (ADR-005).
- **Outliers (ADR-002):** los picos de ruido son señal real (eventos, tráfico, obras) -- no aplicar clipping automático antes de calcular L_Aeq.

### Supuestos específicos — Ruido ambiental

- **L_Aeq,T es energético, no aritmético.** Promediar en dB directamente subestima sistemáticamente el nivel real cuando hay eventos puntuales. Usar siempre `ruido_exceedance_report()`, nunca `series.mean()` para evaluar cumplimiento.
- **Resolución temporal importa.** Con mediciones muy espaciadas (ej. una medición cada varias horas) el L_Aeq calculado es una aproximación pobre del nivel continuo equivalente real -- idealmente sub-horario o continuo.
- **La excepción de fuente natural NO es automática.** `natural_noise=True` es una aserción del analista, no una detección algorítmica -- documentar en el reporte final por qué se invocó esa excepción.
- **Verificación normativa con fecha límite real.** La Ley 2450/2025 puede modificar la Tabla 2 -- revisar `config.NORMA_FUENTES["NORMA_RUIDO"]["fecha_verificacion"]` y su campo `estado` antes de usar estos umbrales en un reporte oficial, sobre todo si `fecha_verificacion` es anterior a la fecha de reglamentación esperada (ver el propio campo `estado` para el estado más reciente conocido).

### Lo que este notebook NO tiene todavía (a diferencia de calidad del aire)

- Conector de datos automatizado (`io/connectors.py`).
- Evidencia empírica propia de este repo sobre desempeño de modelos predictivos para ruido -- los módulos genéricos de `predictive/` se pueden usar igual, pero no hay un benchmark validado como el de PM2.5 (RMSE, HitRate, etc. en `docs/fuentes/calidad_aire.md`).
- Validadores de excedencia con detección algorítmica de fuente natural (Art. 17 Parágrafo 2) -- es y seguirá siendo un criterio humano, no un modelo.


## 8. Cómo adaptar a datos reales

**Paso 1 — Conseguir datos reales**

No hay conector automatizado hoy. Opciones manuales:
```python
# Cargar un CSV con mediciones de sonómetro (fecha, ruido en dB(A))
from estadistica_ambiental.io.loaders import load_csv
df = load_csv("data/raw/ruido_estacion_x.csv", date_col="fecha")
```

**Paso 2 — Validar con rangos físicos**
```python
from estadistica_ambiental.io.validators import validate
val = validate(df, date_col="fecha")
print(val.summary())  # Detecta valores fuera de [0, 140] dB(A)
```

**Paso 3 — Reporte de cumplimiento contra la Tabla 2**
```python
from estadistica_ambiental.inference.intervals import ruido_exceedance_report
reporte = ruido_exceedance_report(df.set_index("fecha")["ruido"], sector="sector_b")
```

**Paso 4 — Sustituir la serie sintética de la sección 1**
- Reemplazar la celda de generación sintética por la carga real.
- Ajustar `SECTOR` según la zona real medida.

---

### Preguntas abiertas para explorar con datos reales
- ¿Qué diferencia hay entre zonas con y sin actividad de construcción activa?
- ¿La excedencia nocturna es sistemáticamente peor que la diurna en zonas residenciales cerca de vías principales?
- ¿Existe una fuente pública real (portal de datos abiertos, CAR específica) con series continuas de ruido que valga la pena conectar automáticamente?

---

### Glosario mínimo
| Término | Definición |
|---|---|
| dB(A) | Decibel con ponderación A -- aproxima la sensibilidad del oído humano por frecuencia. |
| L_Aeq,T | Nivel de presión sonora continuo equivalente ponderado A sobre un intervalo T (Art. 4, Res. 627/2006). |
| Sector (Tabla 2) | Categoría de uso del suelo que determina el estándar aplicable (residencial, comercial, industrial, ecológico, etc.). |

### Referencias normativas
- Resolución 627 de 2006 -- Ministerio de Ambiente, Vivienda y Desarrollo Territorial (hoy MinAmbiente).
- Ley 2450 de 2025 -- "Ley contra el Ruido" (reglamentación derivada pendiente de verificación, ver `config.NORMA_FUENTES`).
